# Kalman Filter as Trend Filter

A trend filter estimates the direction and speed of a price trend (its slope) while ignoring the noise around it. This notebook builds a Kalman filter for that job, compares it with the classic trend filters on real data (GBP/JPY, daily), and tests it in a simple long/flat strategy. The backtests are demonstrations, not validations.

**Contents**

1. Trend = signal + noise: a filter always finds trends, persistence is what matters; market choice
2. The most common trend filters, compared on the same data
3. Kernel shapes: every linear filter is a weighted average of past returns
4. The Kalman filter from scratch: predict and update
5. Kalman gain and the link with the EMA
6. The model: smooth local linear trend, and why automatic estimation fails
7. Kalman vs the classic filters at the same lag
8. From slope to strategy: all filters on the same long/flat rule
9. Filtered vs smoothed: how to build a fake backtest
10. Diagnostics

Requirements: `pip install yfinance statsmodels plotly pandas numpy`

## 0. Setup and data

In [23]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import warnings
warnings.simplefilter("ignore", ConvergenceWarning)

pio.templates.default = "plotly_dark"
# if charts don't show up (some VS Code / JupyterLab versions): pio.renderers.default = "notebook"

def plot_lines(series, title, ytitle=None, xtitle=None, logy=False, height=700):
    fig = go.Figure([go.Scatter(x=s.index, y=s, name=name, line_width=4) for name, s in series.items()])
    fig.update_layout(title=title, xaxis_title=xtitle, yaxis_title=ytitle, height=height, hovermode="x unified")
    if logy:
        # log axis with readable 1-2-5 ticks (plotly's default shows bare digits)
        lo = min(float(s.min()) for s in series.values()); hi = max(float(s.max()) for s in series.values())
        ticks = [m * 10.0**k for k in range(int(np.floor(np.log10(lo))) - 1, int(np.ceil(np.log10(hi))) + 1) for m in (1, 2, 5)]
        ticks = [t for t in ticks if lo / 1.3 <= t <= hi * 1.3]
        fig.update_yaxes(type="log", tickvals=ticks, ticktext=[f"{t:,g}" for t in ticks])
    fig.show()

In [24]:
TICKER = "GBPJPY=X"       # GBP/JPY (chosen in section 1; alternatives: "EURJPY=X", "GC=F", "^GDAXI")
START = "2010-01-01"
BARS_PER_YEAR = 252
COST = 0.0002             # 2 bps per unit of turnover

close = yf.download(TICKER, start=START, auto_adjust=True, progress=False)["Close"].squeeze().dropna()

In [25]:
y = np.log(close)                     # we always filter the log price
ret = y.diff().fillna(0)              # log returns (for the filters)
ret_simple = close.pct_change().fillna(0)   # simple returns (for the P&L)

print(f"{TICKER}: {len(y)} daily bars, {y.index[0].date()} -> {y.index[-1].date()}")
plot_lines({TICKER: close}, f"{TICKER} (log scale)", logy=True)

GBPJPY=X: 4367 daily bars, 2010-01-01 -> 2026-10-09


## 1. Trend = signal + noise

$$\text{price}_t = \text{trend}_t + \text{noise}_t$$

A filter always finds trends, even in pure noise. Below: a random walk with **zero drift** and its 100-bar moving average. Looking back, it seems full of trends.

In [26]:
rng = np.random.default_rng(7)
rw = pd.Series(np.cumsum(rng.normal(0, 0.012, 2500)))       # log price, no drift, no trend by construction
plot_lines({"random walk": rw, "SMA 100": rw.rolling(100).mean()},
           "A random walk 'has trends' too", ytitle="log price", xtitle="bar")

What matters for trading is **persistence**: after a positive trend, are future returns higher than after a negative one? Same test on the random walk and on real data (average 20-bar forward return after a positive or negative 60-bar return, overlapping windows, illustrative only).

In [27]:
def trend_persistence(logp, look=60, fwd=20):
    d = pd.DataFrame({"past": logp.diff(look), "future": logp.shift(-fwd) - logp}).dropna()
    up, down = d.future[d.past > 0].mean(), d.future[d.past < 0].mean()
    return pd.Series({"after up-trend": up, "after down-trend": down, "difference": up - down})

persistence = pd.DataFrame({"random walk": trend_persistence(rw), TICKER: trend_persistence(y)})
(persistence * 100).round(2).astype(str) + " %"

,random walk,GBPJPY=X
after up-trend,-0.31 %,0.14 %
after down-trend,-0.93 %,0.24 %
difference,0.62 %,-0.11 %


### Which market should we use?

A trend filter is only useful if trends **persist**: past returns must be positively related to future returns. We do not pick the market by taste, we measure it first.

The test, on several markets, with the same data source:
- **Persistence** = correlation between the return over the last `L` bars and the return over the next `F` bars. Positive: trends continue. Negative: mean reversion. Near zero: random walk.
- We try several horizons (3, 6, 12 months back; 3, 6, 12 months forward), because a market can be mean reverting at one scale and trending at another.
- Each cell shows two numbers: the correlation with **overlapping** windows (more data, but the samples are not independent), and with **non-overlapping** windows (independent, but few samples). `n` is the number of non-overlapping samples.

How to read it: with `n` independent samples the standard error of a correlation is about $1/\sqrt{n}$. With n = 65 that is 0.12, with n = 32 it is 0.18, with n = 15 it is 0.26. A correlation inside that band is indistinguishable from zero.

In [28]:
CANDIDATES = ["^NDX", "^GDAXI", "EURJPY=X", "USDJPY=X", "GBPJPY=X", "EURUSD=X", "AUDUSD=X", "GC=F", "CL=F"]
GRID = [(63, 63), (126, 63), (126, 126), (252, 63), (252, 126), (252, 252)]     # (look-back, forward) in bars

def persistence_screen(logp, grid=GRID):
    out = {}
    for look, fwd in grid:
        d = pd.DataFrame({"past": logp.diff(look), "future": logp.shift(-fwd) - logp}).dropna()
        nov = d.iloc[::fwd]                                   # non-overlapping samples
        out[f"{look}->{fwd}"] = f"{d.past.corr(d.future):+.2f} / {nov.past.corr(nov.future):+.2f} (n={len(nov)})"
    return out

screen = {}
for t in CANDIDATES:
    px = yf.download(t, start=START, auto_adjust=True, progress=False)["Close"].squeeze().dropna()
    screen[t] = persistence_screen(np.log(px[px > 0]))         # crude oil had negative prices in 2020
pd.DataFrame(screen).T

,63->63,126->63,126->126,252->63,252->126,252->252
^NDX,-0.16 / -0.09 (n=65),-0.06 / -0.06 (n=64),-0.01 / -0.01 (n=32),-0.09 / -0.15 (n=62),-0.10 / -0.14 (n=31),-0.31 / -0.31 (n=15)
^GDAXI,-0.19 / -0.30 (n=66),-0.21 / -0.34 (n=65),-0.21 / -0.21 (n=32),-0.28 / -0.33 (n=63),-0.31 / -0.35 (n=31),-0.40 / -0.48 (n=15)
EURJPY=X,+0.03 / +0.07 (n=68),+0.02 / +0.02 (n=67),+0.05 / +0.07 (n=33),+0.06 / -0.00 (n=65),+0.05 / -0.06 (n=32),-0.06 / +0.07 (n=16)
USDJPY=X,+0.04 / -0.09 (n=68),-0.01 / -0.03 (n=67),-0.05 / +0.06 (n=33),+0.04 / -0.03 (n=65),+0.00 / -0.05 (n=32),-0.02 / +0.00 (n=16)
GBPJPY=X,+0.03 / +0.10 (n=68),+0.08 / +0.11 (n=67),+0.13 / +0.19 (n=33),+0.12 / +0.10 (n=65),+0.14 / +0.10 (n=32),+0.05 / -0.04 (n=16)
EURUSD=X,+0.01 / +0.02 (n=68),+0.03 / -0.01 (n=67),+0.03 / -0.06 (n=33),-0.08 / -0.15 (n=65),-0.20 / -0.34 (n=32),-0.35 / -0.38 (n=16)
AUDUSD=X,-0.17 / -0.21 (n=68),-0.01 / -0.05 (n=67),+0.07 / +0.05 (n=33),-0.08 / -0.09 (n=65),-0.07 / -0.10 (n=32),-0.11 / +0.06 (n=16)
GC=F,+0.07 / +0.07 (n=65),+0.10 / +0.12 (n=64),+0.13 / +0.03 (n=32),+0.16 / +0.10 (n=62),+0.22 / +0.04 (n=31),+0.38 / +0.42 (n=15)
CL=F,-0.28 / -0.27 (n=65),-0.17 / -0.17 (n=64),-0.10 / -0.22 (n=32),-0.17 / -0.19 (n=62),-0.08 / -0.14 (n=31),-0.13 / +0.00 (n=15)


What the screen says, and how we decide:

- **Nasdaq 100 and DAX are mean reverting.** Their correlations are negative at almost every horizon (the Nasdaq is close to zero at 6→6 months), the DAX strongly so. They are the wrong market for a trend filter demo: the filter would read the trend correctly, and the market would then tend to reverse it.
- **Crude oil, EUR/USD and AUD/USD** are mean reverting too, at some horizons.
- **Gold** has a positive correlation that grows with the horizon, but the 12-month figure rests on 15 independent samples, and at 12→6 months the non-overlapping value collapses (+0.23 overlapping, +0.04 non-overlapping).
- **EUR/JPY and USD/JPY** are neutral: a random walk at every scale.
- **GBP/JPY** has the most consistent positive sign: +0.10 to +0.19 (non-overlapping) at every horizon up to 12→6 months. It fades to zero at 12→12 months (n = 16). Each value is within about one standard error of zero, and the simple 60→20 bar test above is also flat (difference -0.11%).

We use **GBP/JPY**, the most consistently positive one. Two honest caveats:
1. No correlation here is statistically strong. With 16 years of data, 9 markets and 6 horizons, some of them look good by chance: picking the best would not survive out of sample. We picked on the *sign and consistency*, not on the highest number.
2. So this notebook shows **how the filter works**, not that GBP/JPY has a tradable edge. The real test is across many markets and out of sample.

## 2. The most common trend filters

For trading we care about the **slope** of the trend (its velocity), expressed as return per bar. The classic estimators:

| Filter | Slope estimate |
| --- | --- |
| Momentum | $(y_t - y_{t-n})/n$ |
| MA crossover | $\frac{2}{n_{slow}-n_{fast}}\left(MA_{fast} - MA_{slow}\right)$ |
| Rolling OLS | slope of a linear regression on the last $n$ prices |
| EMA of returns | exponential average of past returns |

The crossover scaling comes from Bruder et al. (2011): it turns the distance between the two averages into a trend per bar.

To compare them fairly, we give all filters the **same mean lag** (the average age of the information they use, defined in section 3). Here: 30 bars.

In [29]:
def slope_momentum(y, n):
    return (y - y.shift(n)) / n

def slope_crossover(y, n_slow, n_fast=10):
    return 2 * (y.rolling(n_fast).mean() - y.rolling(n_slow).mean()) / (n_slow - n_fast)

def slope_ols(y, n):
    t = np.arange(n) - (n - 1) / 2
    w = t / (t ** 2).sum()
    return y.rolling(n).apply(lambda v: w @ v, raw=True)

def slope_ema(y, span):
    return y.diff().ewm(span=span, adjust=False).mean()

## 3. Kernel shapes: the weights on past returns

Every linear filter can be written as

$$\hat\beta_t = \sum_{k \ge 0} w_k \, r_{t-k}$$

The weights $w_k$ (the **kernel**) are the only thing that distinguishes one filter from another. We measure them directly: we feed the filter a price series with a single return equal to 1 and record how the slope responds over the following bars.

Two numbers summarize a kernel:
- **mean lag** $= \sum k\,w_k / \sum w_k$: how old the information is, on average;
- **noise** $= \sum w_k^2$: the variance of the estimate when returns are pure noise (lower = smoother).

These are the two sides of the lag/noise trade-off.

In [30]:
def kernel(estimator, n_lags=600, burn=2000):
    r = np.zeros(burn + n_lags)
    r[burn] = 1.0
    response = np.asarray(estimator(pd.Series(np.cumsum(r))), float)[burn:]
    return pd.Series(np.nan_to_num(response), name="weight")

def kernel_stats(w):
    k = np.arange(len(w))
    return pd.Series({"sum of weights": w.sum(),
                      "mean lag (bars)": (k * w).sum() / w.sum(),
                      "noise Σw² (x1000)": 1000 * (w ** 2).sum()})

def match_lag(make_estimator, grid, target):
    lags = {p: kernel_stats(kernel(make_estimator(p)))["mean lag (bars)"] for p in grid}
    return min(lags, key=lambda p: abs(lags[p] - target))

TARGET_LAG = 30

n_mom  = match_lag(lambda n: lambda s: slope_momentum(s, n), range(20, 121), TARGET_LAG)
n_xo   = match_lag(lambda n: lambda s: slope_crossover(s, n), range(20, 201), TARGET_LAG)
n_ols  = match_lag(lambda n: lambda s: slope_ols(s, n), range(20, 121), TARGET_LAG)
n_ema  = match_lag(lambda n: lambda s: slope_ema(s, n), range(20, 121), TARGET_LAG)

classic = {
    f"Momentum {n_mom}":        lambda s: slope_momentum(s, n_mom),
    f"Crossover 10/{n_xo}":     lambda s: slope_crossover(s, n_xo),
    f"Rolling OLS {n_ols}":     lambda s: slope_ols(s, n_ols),
    f"EMA of returns {n_ema}":  lambda s: slope_ema(s, n_ema),
}
kernels = {name: kernel(f) for name, f in classic.items()}

plot_lines({k: w.iloc[:150] for k, w in kernels.items()},
           f"Kernel of the classic filters (mean lag ≈ {TARGET_LAG} bars)", ytitle="weight", xtitle="lag (bars)")
pd.DataFrame({k: kernel_stats(w) for k, w in kernels.items()}).T.round(3)

,sum of weights,mean lag (bars),noise Σw² (x1000)
Momentum 61,1.0,30.0,16.393
Crossover 10/83,1.0,30.0,16.075
Rolling OLS 62,1.0,30.0,19.365
EMA of returns 61,1.0,30.0,16.393


- **Momentum**: flat weights, then a hard cut. The oldest return counts as much as the newest, then disappears at once.
- **Crossover**: triangle, as shown in Bruder et al.
- **Rolling OLS**: parabola, still with a finite window.
- **EMA**: highest weight on the latest return, smooth exponential decay, no cut.

All weights sum to 1: on a perfectly linear trend every filter returns the true slope. At the same lag the noise levels are similar: no filter escapes the trade-off.

In [31]:
slopes = pd.DataFrame({name: f(y) * BARS_PER_YEAR for name, f in classic.items()}).dropna()
plot_lines({c: slopes[c] for c in slopes}, f"{TICKER}: estimated slope, annualized", ytitle="trend per year")
print("Correlation between slope estimates:")
slopes.corr().round(3)

Correlation between slope estimates:


,Momentum 61,Crossover 10/83,Rolling OLS 62,EMA of returns 61
Momentum 61,1.000,0.915,0.899,0.843
Crossover 10/83,0.915,1.000,0.918,0.856
Rolling OLS 62,0.899,0.918,1.000,0.702
EMA of returns 61,0.843,0.856,0.702,1.000


## 4. The Kalman filter from scratch

The model (smooth local linear trend), on the log price $y_t$:

$$y_t = \mu_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, R_t) \qquad
\mu_t = \mu_{t-1} + \beta_{t-1} \qquad
\beta_t = \beta_{t-1} + \zeta_t, \quad \zeta_t \sim N(0, Q)$$

- $\mu_t$: hidden trend level; $\beta_t$: slope (velocity)
- $R_t$: noise of the price around the trend; $Q$: how fast the slope can change

At every bar, two steps:

1. **Predict**: $\mu \leftarrow \mu + \beta$. The uncertainty $P$ grows by $Q$.
2. **Update**: surprise $v = y_t - \mu$. Correct level and slope by a fraction of the surprise: $\mu \leftarrow \mu + K_1 v$, $\beta \leftarrow \beta + K_2 v$. The uncertainty shrinks.

The gain $K$ is computed by the filter from $P$ and $R$. With a 2x2 state everything can be written with plain scalars, as in the code below.

In [32]:
def kalman_smooth_trend(y, R, Q):
    # Kalman filter for the smooth trend model. R and Q can be scalars or arrays (time-varying).
    y = np.asarray(y, float)
    n = len(y)
    R, Q = np.broadcast_to(np.asarray(R, float), n), np.broadcast_to(np.asarray(Q, float), n)
    out = {key: np.empty(n) for key in ("level", "slope", "var_slope", "gain_level", "gain_slope", "innov", "innov_var")}
    mu, b = y[0], 0.0
    p11, p12, p22 = 1e6, 0.0, 1e6                        # diffuse start: "I know nothing"

    for t in range(n):
        # predict
        mu += b
        p11, p12, p22 = p11 + 2 * p12 + p22, p12 + p22, p22 + Q[t]
        # update
        S = p11 + R[t]                                   # expected variance of the surprise
        k1, k2 = p11 / S, p12 / S                        # Kalman gain
        v = y[t] - mu                                    # surprise (innovation)
        mu, b = mu + k1 * v, b + k2 * v
        p11, p12, p22 = p11 * (1 - k1), p12 * (1 - k1), p22 - k2 * p12
        for key, val in zip(out, (mu, b, p22, k1, k2, v, S)):
            out[key][t] = val
    return out

### The model in action, on simulated data

To see what the model means, we **generate** a series that follows it exactly, and then look at it the way the filter does:

- the **slope** (bottom) is a slow random walk: it drifts, speeds up, slows down, changes sign. Nobody observes it;
- the **trend level** (top, thick line) is the slope accumulated bar after bar;
- the **observed price** (top, thin line) is the trend plus random noise. This is the only thing the filter sees.

The filter's job is to recover the two hidden lines from the noisy one. Dotted lines are its estimates (using the true noise parameters, skipping the first 30 bars while the filter starts up). The level is tracked closely; the slope is noisier and **lags** the true slope, since it can only learn about a change after seeing it in the prices. This is the lag/noise trade-off again.

In [33]:
rng = np.random.default_rng(28)
n, sd_obs, sd_slope = 400, 0.02, 0.0003            # noise around the trend, slope shock per bar
slope_true = np.cumsum(rng.normal(0, sd_slope, n))            # hidden slope: a slow random walk
level_true = np.cumsum(slope_true)                            # hidden trend: the slope accumulated
price = level_true + rng.normal(0, sd_obs, n)                 # what we observe: trend + noise

sim = kalman_smooth_trend(price, sd_obs**2, sd_slope**2)      # filter run with the true parameters
for k in ("level", "slope"):
    sim[k][:30] = np.nan                                      # skip the start-up phase of the filter

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Observed log price = hidden trend + noise", "Hidden slope (the trend's speed)"))
fig.add_scatter(y=price, name="observed price", line=dict(width=1, color="lightgray"), row=1, col=1)
fig.add_scatter(y=level_true, name="hidden trend level", line=dict(width=2.5), row=1, col=1)
fig.add_scatter(y=sim["level"], name="Kalman level", line=dict(width=1.5, dash="dot"), row=1, col=1)
fig.add_scatter(y=slope_true * BARS_PER_YEAR, name="hidden slope", line=dict(width=2.5), row=2, col=1)
fig.add_scatter(y=sim["slope"] * BARS_PER_YEAR, name="Kalman slope", line=dict(width=1.5, dash="dot"), row=2, col=1)
fig.update_yaxes(title_text="log price", row=1, col=1)
fig.update_yaxes(title_text="trend per year", row=2, col=1)
fig.update_xaxes(title_text="bar", row=2, col=1)
fig.update_layout(height=650, hovermode="x unified").show()

Check against `statsmodels` (with `tolerance = 0`, otherwise statsmodels freezes the gain early to save computation).

In [34]:
R_bar = ret.var()
Q_test = R_bar / 1e5
mine = kalman_smooth_trend(y.values, R_bar, Q_test)

uc = sm.tsa.UnobservedComponents(y.values, level="strend")     # params: irregular, trend
uc.ssm.tolerance = 0
ref = uc.filter([R_bar, Q_test])
print("max |ours - statsmodels| on the slope:", np.abs(mine["slope"][10:] - ref.filtered_state[1][10:]).max())

max |ours - statsmodels| on the slope: 1.2622418400754475e-09


## 5. The Kalman gain, and the link with the EMA

The gain starts high (the filter knows nothing and trusts the data) and converges quickly to a constant. From then on the filter is a **fixed-weight linear filter**.

In [35]:
kf = kalman_smooth_trend(y.values, R_bar, Q_test)
fig = go.Figure()
fig.add_scatter(y=kf["gain_level"][:300], name="gain on level")
fig.add_scatter(y=kf["gain_slope"][:300], name="gain on slope", yaxis="y2")
fig.update_layout(title="Kalman gain converges to a constant", xaxis_title="bar", yaxis_title="gain on level",
                  yaxis2=dict(title="gain on slope", overlaying="y", side="right", showgrid=False), height=650).show()

Bruder et al. show it explicitly for the simplest model (returns = slope + noise, slope = random walk): the Kalman filter **is** an EMA of returns, with smoothing factor equal to the steady-state gain

$$\kappa = \frac{2\sigma_\eta}{\sigma_\eta + \sqrt{\sigma_\eta^2 + 4\sigma_\zeta^2}}$$

Let's verify it numerically.

In [36]:
def kalman_local_level(x, q, r):
    # returns = slope + noise(r), slope = random walk(q)
    m, P, out = 0.0, 1e6, np.empty(len(x))
    for t, xt in enumerate(x):
        P += q
        K = P / (P + r)
        m += K * (xt - m)
        P *= 1 - K
        out[t] = m
    return out

q, r = 1e-3 * R_bar, R_bar
kappa = 2 * np.sqrt(q) / (np.sqrt(q) + np.sqrt(q + 4 * r))
kalman_ll = pd.Series(kalman_local_level(ret.values, q, r), index=ret.index)
ema_equiv = ret.ewm(alpha=kappa, adjust=False).mean()

print(f"steady-state gain = {kappa:.5f}  ->  EMA span ≈ {2 / kappa - 1:.0f} bars")
print("max |Kalman - EMA| after 500 bars:", (kalman_ll - ema_equiv).iloc[500:].abs().max())
plot_lines({"Kalman (local level on returns)": kalman_ll * BARS_PER_YEAR,
            f"EMA, alpha = {kappa:.4f}": ema_equiv * BARS_PER_YEAR},
           "In the simplest model, Kalman = EMA", ytitle="trend per year")

steady-state gain = 0.03113  ->  EMA span ≈ 63 bars
max |Kalman - EMA| after 500 bars: 1.9599538721051628e-10


The two curves overlap exactly. The Kalman filter does not beat the EMA by magic: in the simplest case it *is* an EMA, with the weight derived from the model instead of chosen by eye.

## 6. Our model and its calibration

### Why not let the computer estimate the parameters?

Maximum likelihood (MLE) fits $R$, $q_\mu$, $q_\beta$ by minimizing the **one-bar-ahead** prediction error. But we don't trade one bar ahead: we want a trend over weeks or months. The horizon MLE picks is whatever best predicts the next bar, not the one we want to trade. Bruder et al. find it produces very short-term trends on the S&P 500; on other data it can also degenerate into an almost constant drift. Either way, the horizon is not under our control.

In [37]:
res = sm.tsa.UnobservedComponents(y.values, level="lltrend").fit(disp=False)
print(pd.Series(res.params, index=res.param_names).to_string())

uc_mle = sm.tsa.UnobservedComponents(y.values, level="lltrend")
uc_mle.ssm.tolerance = 0
slope_mle = pd.Series(uc_mle.filter(res.params).filtered_state[1], index=y.index)

def slope_mle_filter(s):
    m = sm.tsa.UnobservedComponents(np.asarray(s, float), level="lltrend")
    m.ssm.tolerance = 0
    return m.filter(res.params).filtered_state[1]

sigma2.irregular    5.940654e-10
sigma2.level        4.798623e-05
sigma2.trend        2.088709e-11


### Calibration from the horizon

Instead we fix one intuitive number, the **trend horizon in bars**, and set

$$\lambda = \frac{R}{Q} = \left(\frac{\text{horizon}}{2\pi}\right)^4$$

A longer horizon means a slope that changes more slowly, so a smoother, slower filter.

In [38]:
def lam(horizon):
    return (horizon / (2 * np.pi)) ** 4

def slope_kalman(y, horizon, R=None):
    R = np.var(np.diff(np.asarray(y, float))) if R is None else R
    return kalman_smooth_trend(y, R, np.mean(R) / lam(horizon))["slope"]

def flips_per_year(s):
    s = s.dropna()
    return (np.sign(s).diff().abs() > 0).sum() / (len(s) / BARS_PER_YEAR)

horizons = (20, 60, 200)
kalman_slopes = {f"horizon {h}": pd.Series(slope_kalman(y.values, h, R_bar), index=y.index) for h in horizons}

plot_lines({"MLE": slope_mle * BARS_PER_YEAR, **{k: s * BARS_PER_YEAR for k, s in kalman_slopes.items()}},
           "Kalman slope: MLE vs horizon-based calibration (annualized)", ytitle="trend per year")

pd.DataFrame({
    "sign changes per year": {"MLE": flips_per_year(slope_mle), **{k: flips_per_year(s) for k, s in kalman_slopes.items()}},
    "mean lag (bars)": {"MLE": kernel_stats(kernel(slope_mle_filter))["mean lag (bars)"], **{f"horizon {h}": kernel_stats(kernel(lambda s, h=h: slope_kalman(s, h, 1.0)))["mean lag (bars)"] for h in horizons}},
}).round(1)

,sign changes per year,mean lag (bars)
MLE,1.7,277.7
horizon 20,23.9,3.6
horizon 60,9.1,12.5
horizon 200,2.5,44.0


Compare the MLE row with the horizon rows: its lag and its number of sign changes are a by-product of one-bar prediction, not a choice. With the horizon we set the lag/noise trade-off directly, and we can match it to the trend scale we want to trade.

## 7. Kalman vs the classic filters at the same lag

We choose the Kalman horizon that gives the same mean lag as the other filters, then compare kernels and noise.

In [39]:
h_match = match_lag(lambda h: lambda s: slope_kalman(s, h, 1.0), range(20, 401, 5), TARGET_LAG)
all_filters = {**classic, f"Kalman horizon {h_match}": lambda s: slope_kalman(s.values, h_match, R_bar)}
kernels[f"Kalman horizon {h_match}"] = kernel(lambda s: slope_kalman(s, h_match, 1.0))

plot_lines({k: w.iloc[:200] for k, w in kernels.items()},
           f"Kernels at the same mean lag (≈ {TARGET_LAG} bars)", ytitle="weight", xtitle="lag (bars)")
pd.DataFrame({k: kernel_stats(w) for k, w in kernels.items()}).T.round(3)

,sum of weights,mean lag (bars),noise Σw² (x1000)
Momentum 61,1.0,30.000,16.393
Crossover 10/83,1.0,30.000,16.075
Rolling OLS 62,1.0,30.000,19.365
EMA of returns 61,1.0,30.000,16.393
Kalman horizon 140,1.0,30.519,15.871


The Kalman kernel is a **damped oscillation**: it rises to a peak on recent lags, decays smoothly, and goes slightly negative on old lags (the filter "corrects" an old trend it no longer believes). No window, no hard cut: an old return fades out gradually instead of dropping out on a single day.

In [40]:
slopes[f"Kalman horizon {h_match}"] = pd.Series(slope_kalman(y.values, h_match, R_bar), index=y.index) * BARS_PER_YEAR
slopes = slopes.dropna()
print("Correlation between slope estimates (including Kalman):")
slopes.corr().round(3)

Correlation between slope estimates (including Kalman):


,Momentum 61,Crossover 10/83,Rolling OLS 62,EMA of returns 61,Kalman horizon 140
Momentum 61,1.000,0.915,0.899,0.843,0.920
Crossover 10/83,0.915,1.000,0.918,0.856,0.970
Rolling OLS 62,0.899,0.918,1.000,0.702,0.961
EMA of returns 61,0.843,0.856,0.702,1.000,0.780
Kalman horizon 140,0.920,0.970,0.961,0.780,1.000


Momentum, crossover, OLS and Kalman are highly correlated (0.90–0.97): at the same lag they see the same trend, with different weights. The EMA of returns is the odd one out (0.70–0.86, and 0.78 with the Kalman): its kernel puts most of the weight on the very latest returns.

## 8. From slope to strategy

The strategy signal is the **t-stat of the slope**: slope divided by its standard deviation from the filter. It becomes informative when the noise $R_t$ changes over time: we set $R_t$ to the EWMA variance of recent returns (past data only), and $Q$ from its one-year rolling mean. When volatility spikes, the filter trusts each bar less and the t-stat shrinks.

Rule for every filter: **long when the signal is above the entry threshold, flat when it falls below the exit threshold.** Signal at the close, position from the next bar, 2 bps cost per trade. For the classic filters the signal is the sign of the slope.

In [41]:
HORIZON, VAR_SPAN, Q_WINDOW = h_match, 30, 250

R_t = ret.pow(2).ewm(span=VAR_SPAN, min_periods=VAR_SPAN).mean().shift(1).bfill()
Q_t = R_t.rolling(Q_WINDOW, min_periods=1).mean() / lam(HORIZON)
kf_t = kalman_smooth_trend(y.values, R_t.values, Q_t.values)
tstat = pd.Series(kf_t["slope"] / np.sqrt(kf_t["var_slope"]), index=y.index)
tstat.iloc[:max(HORIZON, VAR_SPAN)] = np.nan

def long_flat(signal, entry=0.0, exit_=0.0):
    pos, state = np.zeros(len(signal)), 0
    for i, s in enumerate(signal.values):
        if state == 0 and s > entry:
            state = 1
        elif state == 1 and s < exit_:
            state = 0
        pos[i] = state
    return pd.Series(pos, index=signal.index)

def backtest(pos):
    pos = pos.shift(1).fillna(0)
    return pos * ret_simple - COST * pos.diff().abs().fillna(0)

def metrics(r, pos=None):
    eq = (1 + r).cumprod()
    out = {"CAGR": eq.iloc[-1] ** (BARS_PER_YEAR / len(r)) - 1,
           "Vol": r.std() * np.sqrt(BARS_PER_YEAR),
           "Sharpe": r.mean() / r.std() * np.sqrt(BARS_PER_YEAR),
           "MaxDD": (eq / eq.cummax() - 1).min()}
    if pos is not None:
        out["Exposure"] = pos.mean()
        out["Trades/year"] = (pos.diff() > 0).sum() / (len(pos) / BARS_PER_YEAR)
    return pd.Series(out)

In [42]:
start = tstat.first_valid_index()
signals = {name: slopes[name] for name in slopes} | {"Kalman t-stat (R_t)": tstat}
positions = {name: long_flat(s.reindex(y.index)).loc[start:] for name, s in signals.items()}

rets = pd.DataFrame({"Buy & hold": ret_simple.loc[start:]} | {n: backtest(p).loc[start:] for n, p in positions.items()})
table = pd.DataFrame({n: metrics(rets[n], positions.get(n)) for n in rets}).T
plot_lines({c: (1 + rets[c]).cumprod() for c in rets}, "Long/flat equity, same rule for every filter", logy=True)
table.style.format({"CAGR": "{:.1%}", "Vol": "{:.1%}", "Sharpe": "{:.2f}", "MaxDD": "{:.1%}",
                    "Exposure": "{:.0%}", "Trades/year": "{:.1f}"}, na_rep="-")

,CAGR,Exposure,MaxDD,Sharpe,Trades/year,Vol
Buy & hold,2.7%,-,-35.6%,0.30,-,10.8%
Momentum 61,2.2%,61%,-14.7%,0.33,6.3,7.7%
Crossover 10/83,-0.6%,60%,-31.9%,-0.04,2.8,7.7%
Rolling OLS 62,0.4%,59%,-18.8%,0.09,2.2,7.5%
EMA of returns 61,0.8%,59%,-18.0%,0.14,9.6,7.4%
Kalman horizon 140,0.8%,59%,-22.3%,0.14,1.9,7.6%
Kalman t-stat (R_t),0.6%,59%,-20.6%,0.12,2.0,7.7%


Reading the table:
- GBP/JPY had a weak trend over this sample: buy & hold made 2.6% a year with a Sharpe of 0.30. The filters are **not better** on returns: CAGR between -0.6% and 2.2%, Sharpe between -0.04 and 0.33. Only momentum edges out buy & hold on Sharpe (0.33 vs 0.30).
- What the filters buy is **risk reduction**: about 60% exposure, volatility around 7.5% instead of 10.8%, and a shallower maximum drawdown for most of them (from -14.7% to -22.3%, against -35.6%). The crossover is the exception (-31.9%).
- The spread between filters is of the same size as the difference with buy & hold: on one market and one sample, it is noise, not a ranking.
- This is one market, daily bars, fixed parameters, no optimization: a demonstration, not a validation. The value of trend following, if any, shows up across many markets and out of sample, not on one chart.

## 9. Filtered vs smoothed: how to build a fake backtest

- **Filtered**: the estimate at time $t$ uses data up to $t$. The only one you can trade.
- **Smoothed**: the estimate at time $t$ uses the **whole sample**, future included.

Same model, same parameters, same long/flat rule. Only the estimate changes.

In [43]:
params = [R_bar, R_bar / lam(HORIZON)]
uc_s = sm.tsa.UnobservedComponents(y.values, level="strend")
slope_filtered = pd.Series(uc_s.filter(params).filtered_state[1], index=y.index)
slope_smoothed = pd.Series(uc_s.smooth(params).smoothed_state[1], index=y.index)

cheat = pd.DataFrame({"Buy & hold": ret_simple,
                      "Filtered (real)": backtest(long_flat(slope_filtered)),
                      "Smoothed (look-ahead!)": backtest(long_flat(slope_smoothed))}).loc[start:]
plot_lines({c: (1 + cheat[c]).cumprod() for c in cheat}, "Same strategy, filtered vs smoothed", logy=True)
pd.DataFrame({c: metrics(cheat[c]) for c in cheat}).T.style.format({"CAGR": "{:.1%}", "Vol": "{:.1%}", "Sharpe": "{:.2f}", "MaxDD": "{:.1%}"})

,CAGR,Vol,Sharpe,MaxDD
Buy & hold,2.7%,10.8%,0.30,-35.6%
Filtered (real),0.8%,7.6%,0.14,-22.3%
Smoothed (look-ahead!),10.6%,7.9%,1.32,-7.4%


The smoothed curve knows in advance where the trend turns: same model, same rule, but CAGR 10.6% and Sharpe 1.32 (maximum drawdown -7.4%), against 0.8% and 0.14 for the filtered version. Any indicator that uses a smoother, a centered regression or any two-sided filter has the same problem.

## 10. Diagnostics

If the model were exactly right, the standardized surprises $z_t = v_t / \sqrt{S_t}$ would be white noise with variance 1.

In [44]:
z = pd.Series(kf_t["innov"] / np.sqrt(kf_t["innov_var"]), index=y.index).iloc[max(HORIZON, VAR_SPAN):]
print(f"std(z) = {z.std():.2f} (expected ~1)")
print(acorr_ljungbox(z, lags=[5, 10, 20]).round(4).to_string())

ci = 1.96 / np.sqrt(len(z))
fig = go.Figure(go.Bar(x=np.arange(1, 41), y=acf(z, nlags=40)[1:], name="ACF"))
fig.add_hline(y=ci, line_dash="dash", line_color="red")
fig.add_hline(y=-ci, line_dash="dash", line_color="red")
fig.update_layout(title="ACF of standardized innovations (95% bands)", xaxis_title="lag", height=650).show()

std(z) = 2.75 (expected ~1)
       lb_stat  lb_pvalue
5   13939.2224        0.0
10  20271.9536        0.0
20  22829.4707        0.0


The surprises are strongly autocorrelated: in reality the price does not oscillate around a smooth trend with white noise, its deviations are persistent. The model is **wrong as a description of the price**, but it remains a good **filter**. Practical consequence: treat the t-stat as a volatility-scaled signal, not as a statistical test ($t > 2$ does not mean "95% significant trend").

## Takeaways

- Every linear trend filter is a weighted average of past returns; at the same lag they see the same trend.
- The Kalman filter chooses the weights from a model: direct slope, no window, uncertainty, adaptation to volatility. In the simplest case it is exactly an EMA.
- Set the parameters from the horizon, not by MLE, which chases one-bar noise.
- Never use the smoother in a backtest.
- The edge is weak and comes from persistence of trends: it shows up across a portfolio of markets, not in a single chart.